# 06 — Climate VaR (CAPM Beta-Adjusted NGFS Scenarios)

**This notebook is a live, open-source Python implementation of the Climate VaR methodology from my BSc dissertation:**

> *Semar, M. (2025). "Bridging Two Worlds: A Comparative Analysis of Market VaR and Climate VaR Enhanced by Tail-Loss Metrics (LEL/WLEL)." BSc Finance Applied Research Project, Bayes Business School.*

The original dissertation implemented this in EViews on NextEra Energy (NEE) and Caterpillar Inc. (CAT). This notebook reproduces the same methodology in Python, on the project's own ETF portfolio, so it can run on live data end-to-end rather than a one-off academic submission.

**Method:**
1. Estimate each asset's CAPM beta against the market (SPY as market proxy)
2. Apply NGFS transition-scenario market shocks, scaled by beta, to get a firm-specific expected return under each scenario
3. Run a Monte Carlo simulation (mean = scenario-adjusted return, std = historical volatility) to build a simulated return distribution per scenario
4. Extract Climate VaR, LEL and WLEL (quadratic weighting) at 95% and 99% confidence

**Important data note:** `SAMPLE_NGFS_SCENARIOS` in `src/climate_var.py` is an illustrative placeholder, in the same shape as the dissertation's actual NGFS Scenario Explorer download. For a genuinely live/production version, replace it with a fresh download from https://www.ngfs.net/ngfs-scenarios-portal/

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.climate_var import (
    estimate_capm_beta, climate_scenario_shock,
    run_climate_var_scenarios, SAMPLE_NGFS_SCENARIOS
)

returns = pd.read_csv('../data/market_prices.csv', index_col=0, parse_dates=True)
returns = np.log(returns / returns.shift(1)).dropna()
returns.tail()


## 1. Choose a 'market' proxy and an asset to test

The dissertation used the S&P 500 as the market proxy for CAPM. Within this project's own portfolio, `SPY` plays that role naturally; `USO` (oil) is a reasonable first asset to test given its direct exposure to transition risk.

In [ ]:
MARKET_TICKER = 'SPY'
ASSET_TICKER = 'USO'

asset_returns = returns[ASSET_TICKER]
market_returns = returns[MARKET_TICKER]


## 2. Estimate CAPM beta

In [ ]:
beta = estimate_capm_beta(asset_returns, market_returns, risk_free_rate=0.0)
print(f'{ASSET_TICKER} beta vs {MARKET_TICKER}: {beta:.4f}')


## 3. Run all NGFS scenarios through the Climate VaR pipeline

Reproduces the dissertation's Tables 16/17 structure.

In [ ]:
climate_table = run_climate_var_scenarios(
    asset_returns, market_returns,
    scenarios=SAMPLE_NGFS_SCENARIOS,
    confidence_levels=[0.95, 0.99],
    n_simulations=10_000,
)
climate_table.round(2)


## 4. Visualise: Climate VaR and WLEL by scenario

Compare across scenarios — the dissertation's key finding was that the more *indirectly* exposed asset (CAT, an industrial) showed **higher** climate sensitivity than the *directly* exposed one (NEE, a renewables utility), due to supply-chain and regulatory pass-through effects. Worth testing whether a similar pattern holds here (e.g. does TLT or GLD show unexpected climate sensitivity through second-order channels, versus the 'obviously exposed' USO?).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
climate_table[['var_99_pct', 'wlel_99_pct']].plot(kind='bar', ax=ax)
ax.set_ylabel('% of equity value')
ax.set_title(f'{ASSET_TICKER}: Climate VaR vs WLEL by NGFS Scenario (99%)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../results/figures/climate_var_by_scenario.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Repeat across all four portfolio assets

*(Loop the pipeline above over SPY, TLT, GLD, USO — each against a chosen market proxy — to compare climate sensitivity across the whole portfolio, mirroring the dissertation's NEE-vs-CAT comparison but across four assets and four asset classes.)*

In [ ]:
# TODO: loop run_climate_var_scenarios() over each ticker and compare betas + climate VaR


---

**Next notebook:** `07_hybrid_var.ipynb` — the regime-switching Hybrid Market-and-Climate VaR model, blending this notebook's scenario shocks into the short-horizon Market VaR framework.